In [1]:
import pandas as pd
import os
from zipfile import ZipFile
import requests

# ----------------------------
# 1. Download & Extract MovieLens 1M
# ----------------------------
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
dataset_zip = "ml-1m.zip"
extracted_path = "ml-1m"

if not os.path.exists(extracted_path):
    print("Downloading MovieLens 1M dataset...")
    r = requests.get(dataset_url)
    with open(dataset_zip, "wb") as f:
        f.write(r.content)
    with ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download and extraction done.")

# ----------------------------
# 2. Load all files
# ----------------------------
ratings_file = os.path.join(extracted_path, "ratings.dat")
movies_file = os.path.join(extracted_path, "movies.dat")
users_file  = os.path.join(extracted_path, "users.dat")

ratings = pd.read_csv(
    ratings_file, sep="::", engine="python",
    names=["UserID","MovieID","Rating","Timestamp"]
)
movies = pd.read_csv(
    movies_file, sep="::", engine="python",
    names=["MovieID","Title","Genres"], encoding="latin-1"
)
users = pd.read_csv(
    users_file, sep="::", engine="python",
    names=["UserID","Gender","Age","Occupation","Zip-code"]
)

# ----------------------------
# 3. Convert timestamp
# ----------------------------
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

# ----------------------------
# 4. Check for duplicates
# ----------------------------
print("Duplicate rows:")
print(f"Ratings: {ratings.duplicated().sum()}")
print(f"Movies:  {movies.duplicated().sum()}")
print(f"Users:   {users.duplicated().sum()}")

# Drop duplicates
ratings = ratings.drop_duplicates().reset_index(drop=True)
movies  = movies.drop_duplicates().reset_index(drop=True)
users   = users.drop_duplicates().reset_index(drop=True)

# ----------------------------
# 5. Check for missing values in detail
# ----------------------------
def report_missing(df, name):
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"No missing values in {name}.")
    else:
        print(f"Missing values in {name}:")
        for col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                print(f" - Column '{col}': {missing_count} missing")
        print("Rows with missing values:")
        print(df[df.isnull().any(axis=1)])

report_missing(ratings, "ratings")
report_missing(movies, "movies")
report_missing(users, "users")

# ----------------------------
# 6. Filter ratings to May → Dec 2000
# ----------------------------
start_date = '2000-05-01'
end_date   = '2000-12-31'
ratings = ratings[(ratings['Timestamp'] >= start_date) & (ratings['Timestamp'] <= end_date)].reset_index(drop=True)
print("\nRatings after filtering May → Dec 2000:", ratings.shape)


Download and extraction done.
Duplicate rows:
Ratings: 0
Movies:  0
Users:   0
No missing values in ratings.
No missing values in movies.
No missing values in users.

Ratings after filtering May → Dec 2000: (891189, 4)


In [2]:
# ----------------------------
# Create monthly slices
# ----------------------------
monthly_bins = pd.date_range(start=start_date, end='2001-01-01', freq='MS')
ratings['SliceNum'] = pd.cut(ratings['Timestamp'], bins=monthly_bins, right=False).cat.codes + 1

# ----------------------------
# Inspect each slice
# ----------------------------
slice_summary = []

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    num_ratings = slice_df.shape[0]
    num_movies  = slice_df['MovieID'].nunique()
    num_users   = slice_df['UserID'].nunique()
    slice_summary.append({
        'SliceNum': s,
        'Ratings': num_ratings,
        'UniqueMovies': num_movies,
        'UniqueUsers': num_users
    })

slice_summary_df = pd.DataFrame(slice_summary)
print(slice_summary_df)


   SliceNum  Ratings  UniqueMovies  UniqueUsers
0         1    67437          2920          486
1         2    54486          2913          508
2         3    90334          3135          778
3         4   182109          3298         1310
4         5    52421          3083          576
5         6    42294          2993          500
6         7   290793          3552         2357
7         8   111315          3331         1215


In [3]:
import os
import pandas as pd

# ----------------------------
# 1. Create folder for slice metadata
# ----------------------------
os.makedirs("slice_metadata", exist_ok=True)

# ----------------------------
# 2. Split movies into popularity tiers per slice
# ----------------------------
high_pct, mid_pct, low_pct = 0.2, 0.3, 0.5  # 20/30/50 split

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    movie_counts = slice_df.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
    total_movies = len(movie_counts)
    
    # Determine cutoff indices
    high_cut  = int(total_movies * high_pct)
    mid_cut   = high_cut + int(total_movies * mid_pct)
    
    # Assign tiers
    movie_tiers = pd.DataFrame({'MovieID': movie_counts.index})
    movie_tiers['PopTier'] = 'low'  # default
    movie_tiers.loc[:high_cut-1, 'PopTier'] = 'high'
    movie_tiers.loc[high_cut:mid_cut-1, 'PopTier'] = 'medium'
    
    # Save slice metadata
    movie_tiers.to_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv", index=False)
    print(f"Slice {s}: {total_movies} movies → High:{high_cut}, Medium:{mid_cut-high_cut}, Low:{total_movies-mid_cut}")


Slice 1: 2920 movies → High:584, Medium:876, Low:1460
Slice 2: 2913 movies → High:582, Medium:873, Low:1458
Slice 3: 3135 movies → High:627, Medium:940, Low:1568
Slice 4: 3298 movies → High:659, Medium:989, Low:1650
Slice 5: 3083 movies → High:616, Medium:924, Low:1543
Slice 6: 2993 movies → High:598, Medium:897, Low:1498
Slice 7: 3552 movies → High:710, Medium:1065, Low:1777
Slice 8: 3331 movies → High:666, Medium:999, Low:1666


In [4]:
# ----------------------------
# User type assignment per slice
# ----------------------------
import os
import pandas as pd

slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata", exist_ok=True)

threshold = 0.7  # HighProportion threshold to classify mainstream

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty: 
        continue

    # Load movie popularity tiers for this slice
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    movie_tier_map = dict(zip(movie_tiers.MovieID, movie_tiers.PopTier))

    # Map each rating to movie tier
    slice_df['Tier'] = slice_df['MovieID'].map(movie_tier_map)

    # Count number of ratings per tier per user
    tier_counts = slice_df.groupby(['UserID', 'Tier']).size().unstack(fill_value=0)
    # Ensure all columns exist
    for col in ['high', 'medium', 'low']:
        if col not in tier_counts.columns:
            tier_counts[col] = 0

    # Compute HighProportion
    tier_counts['Total'] = tier_counts['high'] + tier_counts['medium'] + tier_counts['low']
    tier_counts['HighProportion'] = tier_counts['high'] / tier_counts['Total']

    # Assign user type
    tier_counts['UserType'] = tier_counts['HighProportion'].apply(
        lambda x: 'mainstream' if x >= threshold else 'niche'
    )

    # Save result
    user_type_df = tier_counts[['UserType']].reset_index()
    user_type_df.to_csv(f"slice_metadata/user_labels_slice_{s}.csv", index=False)
    
    print(f"Slice {s}: Total Users = {len(user_type_df)}, Mainstream = {sum(user_type_df.UserType=='mainstream')}, Niche = {sum(user_type_df.UserType=='niche')}")


Slice 1: Total Users = 486, Mainstream = 229, Niche = 257
Slice 2: Total Users = 508, Mainstream = 245, Niche = 263
Slice 3: Total Users = 778, Mainstream = 334, Niche = 444
Slice 4: Total Users = 1310, Mainstream = 594, Niche = 716
Slice 5: Total Users = 576, Mainstream = 276, Niche = 300
Slice 6: Total Users = 500, Mainstream = 224, Niche = 276
Slice 7: Total Users = 2357, Mainstream = 1420, Niche = 937
Slice 8: Total Users = 1215, Mainstream = 595, Niche = 620


PROJECT README
Temporal Knowledge Graphs for Recommender Systems
================================================

1. What this project is about
-----------------------------

This project studies how user preferences and item popularity change over time,
and how recommender models can adapt to those changes instead of treating data as static.

Most recommender systems:
- Merge all interactions into one dataset
- Assume user behavior and item popularity are fixed
- Ignore temporal structure

That is unrealistic.

In this project, we:
- Split user–item interactions into time slices
- Build a separate Knowledge Graph (KG) for each slice
- Train models slice by slice
- Compare a baseline recommender with a more expressive final model

The goal is to evaluate whether modeling structure and time actually improves recommendations.


2. Why use a Knowledge Graph
----------------------------

A standard user–item matrix only captures:
“User A interacted with Movie B”.

A Knowledge Graph allows us to represent richer structure:
- Users have behavioral types (mainstream vs niche)
- Movies have genres
- Movies have popularity levels
- All of these can change over time

By encoding interactions as a graph:
- Simple graph models can be used as baselines
- Relation-aware models can exploit edge types
- The same data supports multiple model families


3. Why time slicing
-------------------

User behavior in May 2000 is not the same as in December 2000.

Instead of building one global graph, we:
- Split ratings into monthly time slices
- Build one KG per slice
- Allow users and movies to appear, disappear, or change context

This enables:
- Temporal evaluation
- Embedding warm-starting
- Studying preference drift


4. Models trained later
-----------------------

This preprocessing supports two categories of models:

1. Baseline model
   - Treats all edges as the same
   - Uses only graph structure
   - Ignores relation types

2. Final model
   - Uses relation types
   - Distinguishes different kinds of edges
   - Exploits richer KG structure

Both models use the same preprocessed data.


5. Dataset
----------

- MovieLens 1M
- Ratings filtered to May 2000 – December 2000
- Monthly time slices


6. Outputs per time slice
------------------------

For each slice, the following are created:
- Movie popularity tiers
- User behavioral types
- Slice-specific Knowledge Graph
- Node ID mappings for temporal alignment

All outputs are stored as CSV files.


7. Directory structure
----------------------

.
├── slice_metadata/
│   ├── movie_pop_tiers_slice_1.csv
│   ├── user_labels_slice_1.csv
│   ├── ...
│   └── kg/
│       ├── kg_edges_slice_1.csv
│       ├── user_map_slice_1.csv
│       ├── movie_map_slice_1.csv
│       └── ...


8. Movie popularity tiers
-------------------------

Files:
slice_metadata/movie_pop_tiers_slice_{s}.csv

Description:
- Movies are ranked by number of ratings within a slice
- Assigned to popularity tiers:
  - high
  - medium
  - low

These tiers capture short-term popularity, not global popularity.


9. User behavioral types
------------------------

Files:
slice_metadata/user_labels_slice_{s}.csv

Description:
- Users are labeled based on what they consume in that slice
- Users who mostly rate popular movies → mainstream
- Others → niche

User types can change across slices.


10. Knowledge Graphs
--------------------

Each slice has its own Knowledge Graph.


10.1 Edge list
--------------

Files:
slice_metadata/kg/kg_edges_slice_{s}.csv

Columns:
- Src     : source node ID
- Dst     : destination node ID
- RelType : relation type ID

All edges are stored as bidirectional.


10.2 Relation types
-------------------

Relation ID meanings:

0 → User ↔ Movie  
1 → Movie ↔ Genre  
2 → Movie ↔ Popularity Tier  
3 → User ↔ User Type  

Baseline models ignore RelType.
Final models use RelType.


11. Node IDs
------------

Node IDs are:
- Integers
- Unique within a slice
- Assigned in non-overlapping ranges by node type

This guarantees that users, movies, genres, tiers, and user types
cannot be confused with each other.


12. Node ID mappings (important)
--------------------------------

Files:
- slice_metadata/kg/user_map_slice_{s}.csv
- slice_metadata/kg/movie_map_slice_{s}.csv

These map original UserID / MovieID to slice-local node IDs.

They are required for:
- Matching entities across slices
- Warm-starting embeddings
- Temporal stabilization during training


13. What is NOT stored
----------------------

- No trained models
- No tensors
- No pickled Python objects
- No framework-specific formats

Everything is reconstructed from CSVs in training notebooks.


14. How a new training session should start
-------------------------------------------

A fresh session should:
1. Select a slice
2. Load KG edges and mappings
3. Convert data to tensors as needed
4. Train baseline and final models
5. Optionally align embeddings across slices


15. Summary
-----------

This project builds time-aware Knowledge Graphs from MovieLens data
to compare a baseline recommender against a relation-aware model
while explicitly modeling temporal dynamics.


K = 100

In [6]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp
import copy

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=100):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u, final_i = torch.split(final_emb, [num_users, len(item_map)])
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': get_gini(item_counts), 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (KG Structure)
# ----------------------------
class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)

    def forward(self, adj):
        x = self.emb.weight
        embs = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(adj, x)
            embs.append(x)
        return torch.mean(torch.stack(embs), dim=0)

def build_adjacency(u_list, i_list, n_users, n_items):
    R = sp.coo_matrix((np.ones(len(u_list)), (u_list, i_list)), shape=(n_users, n_items))
    i = torch.LongTensor(np.vstack((R.row, R.col)))
    edge_index = torch.stack([torch.cat([i[0], i[1]+n_users]), torch.cat([i[1]+n_users, i[0]])])
    adj = torch.sparse_coo_tensor(edge_index, torch.ones(edge_index.shape[1]), (n_users+n_items,n_users+n_items)).coalesce()
    deg = torch.sparse.sum(adj, dim=1).to_dense().pow(-0.5)
    deg[deg == float('inf')] = 0
    return torch.sparse_coo_tensor(edge_index, deg[edge_index[0]]*deg[edge_index[1]], (n_users+n_items,n_users+n_items)).to(device)

# ----------------------------
# 4. Training Loop (Fairness Removed)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048

prev_model_state = None
prev_u_map, prev_i_map = None, None

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u:i for i,u in enumerate(train_df['UserID'].unique())}
    i_map = {m:i for i,m in enumerate(train_df['MovieID'].unique())}
    num_u, num_i = len(u_map), len(i_map)
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, num_u, num_i)
    
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model = LightGCN(num_u + num_i).to(device)
    
    # --- TEMPORAL STABILIZATION (Warm-start) ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            for uid, n_idx in u_map.items():
                if uid in prev_u_map: 
                    model.emb.weight.data[n_idx] = val_device[prev_u_map[uid]]
            for mid, n_idx in i_map.items():
                if mid in prev_i_map: 
                    model.emb.weight.data[n_idx + num_u] = val_device[prev_i_map[mid] + len(prev_u_map)]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            
            # Vanilla BPR Loss (No weight adjustments)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Calculate and output metrics
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        # Note: Weights are constant/neutral in vanilla training, but output format is kept as requested
        print(f"Popularity exposure (percent): {metrics['PopExposure']} weights {{ mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }}\n")
        
        # Save state for the next slice
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_u_map, prev_i_map = u_map, i_map

Using device: cpu
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.2694    | 0.1931    | 0.0238    | 0.0251    | 0.9269
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.00%', 'high': '100.00%'}, 'niche': {'low': '1.74%', 'medium': '2.02%', 'high': '96.24%'}} weights { mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }

2 -> 3        | 117    | 0.2053    | 0.1564    | 0.1138    | 0.0841    | 0.8918
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.24%', 'high': '99.76%'}, 'niche': {'low': '4.61%', 'medium': '8.25%', 'high': '87.14%'}} weights { mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }

3 -> 4        | 166    | 0.2490    | 0.1578    | 0.0482    | 0.0987    | 0.8583
Popularity exposure (percent): {'mainstream': {'low': '1.61%', 'medium': '1.61%', 'high': '96.77%'}, 'niche': {'l

K = 75

In [7]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp
import copy

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=75):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u, final_i = torch.split(final_emb, [num_users, len(item_map)])
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': get_gini(item_counts), 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (KG Structure)
# ----------------------------
class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)

    def forward(self, adj):
        x = self.emb.weight
        embs = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(adj, x)
            embs.append(x)
        return torch.mean(torch.stack(embs), dim=0)

def build_adjacency(u_list, i_list, n_users, n_items):
    R = sp.coo_matrix((np.ones(len(u_list)), (u_list, i_list)), shape=(n_users, n_items))
    i = torch.LongTensor(np.vstack((R.row, R.col)))
    edge_index = torch.stack([torch.cat([i[0], i[1]+n_users]), torch.cat([i[1]+n_users, i[0]])])
    adj = torch.sparse_coo_tensor(edge_index, torch.ones(edge_index.shape[1]), (n_users+n_items,n_users+n_items)).coalesce()
    deg = torch.sparse.sum(adj, dim=1).to_dense().pow(-0.5)
    deg[deg == float('inf')] = 0
    return torch.sparse_coo_tensor(edge_index, deg[edge_index[0]]*deg[edge_index[1]], (n_users+n_items,n_users+n_items)).to(device)

# ----------------------------
# 4. Training Loop (Fairness Removed)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048

prev_model_state = None
prev_u_map, prev_i_map = None, None

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u:i for i,u in enumerate(train_df['UserID'].unique())}
    i_map = {m:i for i,m in enumerate(train_df['MovieID'].unique())}
    num_u, num_i = len(u_map), len(i_map)
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, num_u, num_i)
    
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model = LightGCN(num_u + num_i).to(device)
    
    # --- TEMPORAL STABILIZATION (Warm-start) ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            for uid, n_idx in u_map.items():
                if uid in prev_u_map: 
                    model.emb.weight.data[n_idx] = val_device[prev_u_map[uid]]
            for mid, n_idx in i_map.items():
                if mid in prev_i_map: 
                    model.emb.weight.data[n_idx + num_u] = val_device[prev_i_map[mid] + len(prev_u_map)]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            
            # Vanilla BPR Loss (No weight adjustments)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Calculate and output metrics
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        # Note: Weights are constant/neutral in vanilla training, but output format is kept as requested
        print(f"Popularity exposure (percent): {metrics['PopExposure']} weights {{ mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }}\n")
        
        # Save state for the next slice
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_u_map, prev_i_map = u_map, i_map

Using device: cpu
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.2157    | 0.1808    | 0.0168    | 0.0222    | 0.9412
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.00%', 'high': '100.00%'}, 'niche': {'low': '1.68%', 'medium': '1.65%', 'high': '96.67%'}} weights { mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }

2 -> 3        | 117    | 0.1781    | 0.1497    | 0.1053    | 0.0753    | 0.9103
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '0.00%', 'high': '100.00%'}, 'niche': {'low': '4.54%', 'medium': '6.76%', 'high': '88.70%'}} weights { mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }

3 -> 4        | 166    | 0.1835    | 0.1426    | 0.0316    | 0.0893    | 0.8798
Popularity exposure (percent): {'mainstream': {'low': '1.61%', 'medium': '1.05%', 'high': '97.33%'}, 'niche': {'

K = 50

In [ ]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp
import copy

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=50):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u, final_i = torch.split(final_emb, [num_users, len(item_map)])
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': get_gini(item_counts), 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (KG Structure)
# ----------------------------
class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)

    def forward(self, adj):
        x = self.emb.weight
        embs = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(adj, x)
            embs.append(x)
        return torch.mean(torch.stack(embs), dim=0)

def build_adjacency(u_list, i_list, n_users, n_items):
    R = sp.coo_matrix((np.ones(len(u_list)), (u_list, i_list)), shape=(n_users, n_items))
    i = torch.LongTensor(np.vstack((R.row, R.col)))
    edge_index = torch.stack([torch.cat([i[0], i[1]+n_users]), torch.cat([i[1]+n_users, i[0]])])
    adj = torch.sparse_coo_tensor(edge_index, torch.ones(edge_index.shape[1]), (n_users+n_items,n_users+n_items)).coalesce()
    deg = torch.sparse.sum(adj, dim=1).to_dense().pow(-0.5)
    deg[deg == float('inf')] = 0
    return torch.sparse_coo_tensor(edge_index, deg[edge_index[0]]*deg[edge_index[1]], (n_users+n_items,n_users+n_items)).to(device)

# ----------------------------
# 4. Training Loop (Fairness Removed)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048

prev_model_state = None
prev_u_map, prev_i_map = None, None

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u:i for i,u in enumerate(train_df['UserID'].unique())}
    i_map = {m:i for i,m in enumerate(train_df['MovieID'].unique())}
    num_u, num_i = len(u_map), len(i_map)
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, num_u, num_i)
    
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model = LightGCN(num_u + num_i).to(device)
    
    # --- TEMPORAL STABILIZATION (Warm-start) ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            for uid, n_idx in u_map.items():
                if uid in prev_u_map: 
                    model.emb.weight.data[n_idx] = val_device[prev_u_map[uid]]
            for mid, n_idx in i_map.items():
                if mid in prev_i_map: 
                    model.emb.weight.data[n_idx + num_u] = val_device[prev_i_map[mid] + len(prev_u_map)]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            
            # Vanilla BPR Loss (No weight adjustments)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Calculate and output metrics
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        # Note: Weights are constant/neutral in vanilla training, but output format is kept as requested
        print(f"Popularity exposure (percent): {metrics['PopExposure']} weights {{ mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }}\n")
        
        # Save state for the next slice
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_u_map, prev_i_map = u_map, i_map

K  = 25

In [ ]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp
import copy

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=25):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u, final_i = torch.split(final_emb, [num_users, len(item_map)])
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': get_gini(item_counts), 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (KG Structure)
# ----------------------------
class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)

    def forward(self, adj):
        x = self.emb.weight
        embs = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(adj, x)
            embs.append(x)
        return torch.mean(torch.stack(embs), dim=0)

def build_adjacency(u_list, i_list, n_users, n_items):
    R = sp.coo_matrix((np.ones(len(u_list)), (u_list, i_list)), shape=(n_users, n_items))
    i = torch.LongTensor(np.vstack((R.row, R.col)))
    edge_index = torch.stack([torch.cat([i[0], i[1]+n_users]), torch.cat([i[1]+n_users, i[0]])])
    adj = torch.sparse_coo_tensor(edge_index, torch.ones(edge_index.shape[1]), (n_users+n_items,n_users+n_items)).coalesce()
    deg = torch.sparse.sum(adj, dim=1).to_dense().pow(-0.5)
    deg[deg == float('inf')] = 0
    return torch.sparse_coo_tensor(edge_index, deg[edge_index[0]]*deg[edge_index[1]], (n_users+n_items,n_users+n_items)).to(device)

# ----------------------------
# 4. Training Loop (Fairness Removed)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048

prev_model_state = None
prev_u_map, prev_i_map = None, None

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u:i for i,u in enumerate(train_df['UserID'].unique())}
    i_map = {m:i for i,m in enumerate(train_df['MovieID'].unique())}
    num_u, num_i = len(u_map), len(i_map)
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, num_u, num_i)
    
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model = LightGCN(num_u + num_i).to(device)
    
    # --- TEMPORAL STABILIZATION (Warm-start) ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            for uid, n_idx in u_map.items():
                if uid in prev_u_map: 
                    model.emb.weight.data[n_idx] = val_device[prev_u_map[uid]]
            for mid, n_idx in i_map.items():
                if mid in prev_i_map: 
                    model.emb.weight.data[n_idx + num_u] = val_device[prev_i_map[mid] + len(prev_u_map)]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            
            # Vanilla BPR Loss (No weight adjustments)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Calculate and output metrics
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        # Note: Weights are constant/neutral in vanilla training, but output format is kept as requested
        print(f"Popularity exposure (percent): {metrics['PopExposure']} weights {{ mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }}\n")
        
        # Save state for the next slice
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_u_map, prev_i_map = u_map, i_map

K = 10

In [ ]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp
import copy

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=10):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u, final_i = torch.split(final_emb, [num_users, len(item_map)])
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': get_gini(item_counts), 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (KG Structure)
# ----------------------------
class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=3):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)

    def forward(self, adj):
        x = self.emb.weight
        embs = [x]
        for _ in range(self.n_layers):
            x = torch.sparse.mm(adj, x)
            embs.append(x)
        return torch.mean(torch.stack(embs), dim=0)

def build_adjacency(u_list, i_list, n_users, n_items):
    R = sp.coo_matrix((np.ones(len(u_list)), (u_list, i_list)), shape=(n_users, n_items))
    i = torch.LongTensor(np.vstack((R.row, R.col)))
    edge_index = torch.stack([torch.cat([i[0], i[1]+n_users]), torch.cat([i[1]+n_users, i[0]])])
    adj = torch.sparse_coo_tensor(edge_index, torch.ones(edge_index.shape[1]), (n_users+n_items,n_users+n_items)).coalesce()
    deg = torch.sparse.sum(adj, dim=1).to_dense().pow(-0.5)
    deg[deg == float('inf')] = 0
    return torch.sparse_coo_tensor(edge_index, deg[edge_index[0]]*deg[edge_index[1]], (n_users+n_items,n_users+n_items)).to(device)

# ----------------------------
# 4. Training Loop (Fairness Removed)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048

prev_model_state = None
prev_u_map, prev_i_map = None, None

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u:i for i,u in enumerate(train_df['UserID'].unique())}
    i_map = {m:i for i,m in enumerate(train_df['MovieID'].unique())}
    num_u, num_i = len(u_map), len(i_map)
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, num_u, num_i)
    
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model = LightGCN(num_u + num_i).to(device)
    
    # --- TEMPORAL STABILIZATION (Warm-start) ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            for uid, n_idx in u_map.items():
                if uid in prev_u_map: 
                    model.emb.weight.data[n_idx] = val_device[prev_u_map[uid]]
            for mid, n_idx in i_map.items():
                if mid in prev_i_map: 
                    model.emb.weight.data[n_idx + num_u] = val_device[prev_i_map[mid] + len(prev_u_map)]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            
            # Vanilla BPR Loss (No weight adjustments)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Calculate and output metrics
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        # Note: Weights are constant/neutral in vanilla training, but output format is kept as requested
        print(f"Popularity exposure (percent): {metrics['PopExposure']} weights {{ mainstream: 1.0/1.0/1.0, niche :1.0/1.0/1.0 }}\n")
        
        # Save state for the next slice
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_u_map, prev_i_map = u_map, i_map